# Week 5: Supporting Calculations for the Final Business Report
Reproduces the numbers in the report that were not produced in the Week 2 to 4 notebooks.
Dataset: APY (India Data Portal), cleaned in Week 2 (450,350 rows, with is_outlier and crop_failure flags).

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:\PROJECT\Yuva Intern\Datasets\cleaned_apy_data.csv")
print(df.shape)
print(df.columns.tolist())

print("Outliers:", df['is_outlier'].sum())
print("Crop failures:", df['crop_failure'].sum())
print("Both flags:", (df['is_outlier'] & df['crop_failure']).sum())
print("Either flag:", (df['is_outlier'] | df['crop_failure']).sum())

(450350, 18)
['id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit', 'is_outlier', 'crop_failure']
Outliers: 14453
Crop failures: 3997
Both flags: 2
Either flag: 18448


## 1. The 2014-15 Wheat Shock: National Totals
Compares 2013-14 with 2014-15 using total area, total production and area-weighted yield (total production divided by total area).

In [3]:
df['year_start'] = df['year'].str[:4].astype(int)
wheat = df[df['crop_name'] == 'Wheat']

for yr in [2013, 2014]:
    d = wheat[wheat['year_start'] == yr]
    print(yr,
          "| area (million ha):", round(d['area'].sum() / 1e6, 2),
          "| production (million t):", round(d['production'].sum() / 1e6, 2),
          "| weighted yield (t/ha):", round(d['production'].sum() / d['area'].sum(), 3))

2013 | area (million ha): 31.62 | production (million t): 102.97 | weighted yield (t/ha): 3.257
2014 | area (million ha): 31.19 | production (million t): 84.96 | weighted yield (t/ha): 2.724


In [4]:
p13 = wheat[wheat['year_start'] == 2013]['production'].sum()
p14 = wheat[wheat['year_start'] == 2014]['production'].sum()
print(round((p14 / p13 - 1) * 100, 1))

-17.5


## 2. Which States Drove the Fall? (2013-14 vs 2014-15)
Area-weighted wheat yield and production change by state, for states with over 500,000 tonnes of production in 2013-14.

In [5]:
def state_table(yr):
    d = wheat[wheat['year_start'] == yr].groupby('state_name')[['area', 'production']].sum()
    d['yield'] = d['production'] / d['area']
    return d

s13 = state_table(2013)
s14 = state_table(2014)

change = s13.join(s14, lsuffix='_13', rsuffix='_14')
change['yield_change_pct'] = (change['yield_14'] / change['yield_13'] - 1) * 100
change['prod_change_mt'] = (change['production_14'] - change['production_13']) / 1e6

net = (s14['production'].sum() - s13['production'].sum()) / 1e6
change['share_of_net_loss_pct'] = change['prod_change_mt'] / net * 100

big = change[change['production_13'] > 500000].sort_values('prod_change_mt')
print('Net national change (million t):', round(net, 2))
print(big[['yield_13', 'yield_14', 'yield_change_pct', 'prod_change_mt', 'share_of_net_loss_pct']].round(2))

Net national change (million t): -18.01
                yield_13  yield_14  yield_change_pct  prod_change_mt  \
state_name                                                             
Uttar Pradesh       3.20      2.04            -36.37          -11.44   
Punjab              5.02      4.29            -14.42           -2.57   
Bihar               2.85      1.66            -41.95           -2.56   
Haryana             4.72      3.98            -15.73           -1.34   
Gujarat             3.19      2.81            -12.00           -1.31   
Rajasthan           3.44      2.96            -13.88           -1.20   
Uttarakhand         2.47      1.93            -22.00           -0.19   
Maharashtra         1.44      1.23            -14.88           -0.17   
West Bengal         2.80      2.81              0.27            0.01   
Madhya Pradesh      2.57      3.08             19.86            2.74   

                share_of_net_loss_pct  
state_name                             
Uttar Pradesh  

## 3. Inside Uttar Pradesh: How Many Districts Fell?
Compares each district's Rabi wheat yield between 2013-14 and 2014-15, using districts reported in both years.

In [6]:
up = wheat[(wheat['state_name'] == 'Uttar Pradesh') & (wheat['season'] == 'Rabi')]

y13 = up[up['year_start'] == 2013].set_index('district_name')['yield']
y14 = up[up['year_start'] == 2014].set_index('district_name')['yield']

districts = pd.concat([y13, y14], axis=1, keys=['y13', 'y14']).dropna()
districts['change_pct'] = (districts['y14'] / districts['y13'] - 1) * 100

print("Districts compared:", len(districts))
print("Districts with lower yield:", (districts['change_pct'] < 0).sum())
print("Median change (%):", round(districts['change_pct'].median(), 1))
print("Down more than 20%:", (districts['change_pct'] < -20).sum())
print("Down more than 40%:", (districts['change_pct'] < -40).sum())
print(districts.sort_values('change_pct').head(5).round(1))

Districts compared: 75
Districts with lower yield: 74
Median change (%): -34.9
Down more than 20%: 67
Down more than 40%: 30
               y13  y14  change_pct
district_name                      
Kannauj        3.8  1.4       -63.2
Kanpur Nagar   3.8  1.5       -60.7
Jalaun         3.0  1.3       -55.4
Etawah         3.7  1.7       -54.3
Kanpur Dehat   3.6  1.7       -53.5


## 4. Was It Only Wheat? Other Rabi Crops in Uttar Pradesh
If every winter crop fell, the cause is more likely season-wide (such as weather) than specific to wheat.

In [7]:
rabi_crops = ['Wheat', 'Barley', 'Gram', 'Rapeseed &Mustard', 'Masoor']

for crop in rabi_crops:
    d = df[(df['crop_name'] == crop) & (df['state_name'] == 'Uttar Pradesh') & (df['season'] == 'Rabi')]
    a = d[d['year_start'] == 2013]
    b = d[d['year_start'] == 2014]
    y_old = a['production'].sum() / a['area'].sum()
    y_new = b['production'].sum() / b['area'].sum()
    print(crop, ':', round((y_new / y_old - 1) * 100, 1), '%')

Wheat : -36.4 %
Barley : -41.3 %
Gram : -31.1 %
Rapeseed &Mustard : -23.4 %
Masoor : -22.3 %


## 5. How Much of the All-Crop Yield Rise Is Just Crop Mix?
Tests whether the apparent rise in all-crop yield (1997 to 2022) is caused by changes in which crops are reported, by holding the crop mix fixed at its 1997 shares.

In [8]:
ok = df[(~df['is_outlier']) & (~df['crop_failure'])]

observed = ok.groupby('year_start')['yield'].mean()

base_mix = ok[ok['year_start'] == 1997]['crop_name'].value_counts(normalize=True)
crop_means = ok.groupby(['year_start', 'crop_name'])['yield'].mean().unstack()

fixed = {}
for yr in crop_means.index:
    row = crop_means.loc[yr].reindex(base_mix.index)
    w = base_mix[row.notna()]
    fixed[yr] = (row.dropna() * w / w.sum()).sum()
fixed = pd.Series(fixed)

print("Observed all-crop mean, 1997 -> 2022: %.1f -> %.1f (%+.0f%%)" % (observed[1997], observed[2022], (observed[2022] / observed[1997] - 1) * 100))
print("Fixed 1997 crop mix, 1997 -> 2022: %.1f -> %.1f (%+.0f%%)" % (fixed[1997], fixed[2022], (fixed[2022] / fixed[1997] - 1) * 100))
print("Share of the rise due to crop mix: %.0f%%" % ((observed[2022] - fixed[2022]) / (observed[2022] - observed[1997]) * 100))

Observed all-crop mean, 1997 -> 2022: 45.6 -> 81.7 (+79%)
Fixed 1997 crop mix, 1997 -> 2022: 45.6 -> 65.9 (+45%)
Share of the rise due to crop mix: 44%


## 6. Sugarcane's Low-Yield Cluster
Counts Sugarcane records below 5 t/ha, checks how many were flagged in Week 2, and shows which states they come from.

In [9]:
sc = df[df['crop_name'] == 'Sugarcane'].copy()
sc['low'] = sc['yield'] < 5

print("Sugarcane records below 5 t/ha:", sc['low'].sum(), "of", len(sc), "(%.1f%%)" % (sc['low'].mean() * 100))
print("  of which flagged:", (sc['low'] & (sc['is_outlier'] | sc['crop_failure'])).sum())

by_state = sc.groupby('state_name')['low'].agg(['sum', 'count', 'mean'])
by_state['share_pct'] = by_state['mean'] * 100
print(by_state.sort_values('sum', ascending=False).head(6).round(1))

Sugarcane records below 5 t/ha: 869 of 12367 (7.0%)
  of which flagged: 42
                   sum  count  mean  share_pct
state_name                                    
Chhattisgarh       361    469   0.8       77.0
Madhya Pradesh     153   1061   0.1       14.4
Meghalaya          151    151   1.0      100.0
Mizoram             51    176   0.3       29.0
Himachal Pradesh    38    147   0.3       25.9
Jammu And Kashmir   37     44   0.8       84.1


## 7. Recovery After the Shock
Area-weighted wheat yield by year, nationally and for Uttar Pradesh, to show how quickly yields recovered after 2014-15.

In [10]:
tot = wheat.groupby('year_start')[['area', 'production']].sum()
tot['weighted_yield'] = tot['production'] / tot['area']

up_w = wheat[wheat['state_name'] == 'Uttar Pradesh'].groupby('year_start')[['area', 'production']].sum()
up_w['weighted_yield'] = up_w['production'] / up_w['area']

recovery = pd.DataFrame({'national': tot['weighted_yield'], 'uttar_pradesh': up_w['weighted_yield']})
print(recovery.loc[2012:2017].round(3))

            national  uttar_pradesh
year_start                         
2012           3.314          3.219
2013           3.257          3.201
2014           2.724          2.037
2015           3.112          2.786
2016           3.572          3.538
2017           3.647          3.655


## Summary of Results Reproduced in This Notebook

| Report item | Result |
|---|---|
| Records flagged (either flag) | 18,448 of 450,350 (2 records carry both flags) |
| 2014-15 national wheat | Area -1.4%, production 102.97 to 84.96 million t (-17.5%), weighted yield 3.257 to 2.724 t/ha |
| Uttar Pradesh share of net loss | 63.5% (yield -36.4%) |
| UP districts with lower yield | 74 of 75; median fall 34.9% |
| Other UP rabi crops | Barley -41.3%, Gram -31.1%, Mustard -23.4%, Masoor -22.3% |
| All-crop yield rise explained by crop mix | 44% (+79% observed, +45% with 1997 mix) |
| Sugarcane below 5 t/ha | 869 of 12,367 (7.0%); only 42 flagged in Week 2 |
| Recovery | UP weighted yield 2.04 (2014-15), 2.79 (2015-16), 3.54 (2016-17) |